# Allscripts Touchworks — Note Hydration

Populates `_exponent.omop_tw.note` from Allscripts Touchworks clinical result/report notes.

## Source Tables
- `_bronze_allscripts_tw_works.dbo_order_activity_header` — note metadata (patient, encounter, visit, date)
- `_bronze_allscripts_tw_works.dbo_order_result_mapper` — links order activity header to result activity header
- `_bronze_allscripts_tw_works.dbo_result_text` — note body text

## Join Chain
`dbo_order_activity_header.ID` → `dbo_order_result_mapper.OrderActivityHeaderID`
→ `dbo_order_result_mapper.ResultActivityHeaderID` = `dbo_result_text.ResultID`

## Pipeline
1. `silver_note` — staged temp view, full OMOP field set
2. MERGE → `omop_silver.note`
3. INSERT → `omop_mapping.source_to_note`
4. `gold` — resolves surrogate IDs and FK references
5. MERGE → `omop_tw.note`

## Dependencies
- `omop_mapping.source_to_person` must be populated for allscripts_tw
- `omop_mapping.source_to_visit_occurrence` must be populated for allscripts_tw

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_allscripts.note;

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_tw.note;

In [0]:
%sql
-- DELETE FROM _exponent.omop_silver.note
-- WHERE source_system = 'allscripts_tw';

In [0]:
%sql
-- DELETE FROM _exponent.omop_mapping.source_to_note
-- WHERE source_system = 'allscripts_tw';

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_note AS

WITH report_ahs_mtemplate_deduped AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY ID
      ORDER BY
        ClinicalDTTM DESC,
        PerformedDTTM DESC,
        CreateDTTM DESC
    ) AS row_number
  FROM _exponent._bronze_allscripts_tw.report_ahs_mtemplate
),

report_ahs_chart_viewer_deduped AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY
        PatientID,
        EncounterID,
        DocumentTypeID
      ORDER BY
        LastUpdateDTTM DESC,
        AuthoredDTTM DESC
    ) AS row_number
  FROM _exponent._bronze_allscripts_tw.report_ahs_chart_viewer
),

dbo_vendor_item_deduped AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY ID
      ORDER BY
        PerformedDTTM DESC,
        RecordedDTTM DESC,
        CreateDTTM DESC
    ) AS row_number
  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_vendor_item
),

dbo_vendor_item_extension_deduped AS (
  SELECT
    VendorItemID,
    MAX(ExtensionValue) AS ExtensionValue
  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_vendor_item_extension
  WHERE ExtensionType IN ('AddedText', 'DESC')
    AND ExtensionValue IS NOT NULL
    AND TRIM(ExtensionValue) <> ''
  GROUP BY VendorItemID
)

SELECT
  CONCAT_WS(
    CHR(31),
    'allscripts_tw',
    'report_ahs_mtemplate',
    'ID',
    CAST(report_ahs_mtemplate_deduped.ID AS BIGINT)
  ) AS note_source_value,

  source_to_person.person_id,

  CAST(
    COALESCE(
      report_ahs_mtemplate_deduped.ClinicalDTTM,
      report_ahs_mtemplate_deduped.PerformedDTTM,
      report_ahs_mtemplate_deduped.CreateDTTM
    ) AS DATE
  ) AS note_date,

  COALESCE(
    report_ahs_mtemplate_deduped.ClinicalDTTM,
    report_ahs_mtemplate_deduped.PerformedDTTM,
    report_ahs_mtemplate_deduped.CreateDTTM
  ) AS note_datetime,

  32817 AS note_type_concept_id,
  3030653 AS note_class_concept_id,

  COALESCE(
    report_ahs_chart_viewer_deduped.DisplayName,
    report_ahs_mtemplate_deduped.DocumentName,
    report_ahs_mtemplate_deduped.NoteSectionName
  ) AS note_title,

  CONCAT_WS(
    '\n',
    CONCAT(
      'Question: ',
      COALESCE(
        report_ahs_mtemplate_deduped.medcinfinding,
        report_ahs_mtemplate_deduped.DisplayName,
        report_ahs_mtemplate_deduped.QO_DE
      )
    ),
    CONCAT(
      'Answer: ',
      COALESCE(
        report_ahs_mtemplate_deduped.answer,
        dbo_vendor_item_extension_deduped.ExtensionValue
      )
    )
  ) AS note_text,

  32678 AS encoding_concept_id,
  4180186 AS language_concept_id,

  source_to_provider.provider_id,

  source_to_visit_occurrence.visit_occurrence_id,

  NULL AS visit_detail_id,
  NULL AS note_event_id,
  NULL AS note_event_field_concept_id,

  'allscripts_tw' AS source_system,
  CURRENT_TIMESTAMP() AS last_mod_tsp

FROM _exponent._bronze_allscripts_tw_works_vw.dbo_visit

JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter
  ON dbo_encounter.visitid = dbo_visit.id

JOIN report_ahs_mtemplate_deduped
  ON CAST(report_ahs_mtemplate_deduped.encounterID AS BIGINT) =
     CAST(dbo_encounter.id AS BIGINT)
 AND report_ahs_mtemplate_deduped.row_number = 1

LEFT JOIN report_ahs_chart_viewer_deduped
  ON CAST(report_ahs_chart_viewer_deduped.PatientID AS BIGINT) =
     CAST(report_ahs_mtemplate_deduped.patientid AS BIGINT)
 AND CAST(report_ahs_chart_viewer_deduped.EncounterID AS BIGINT) =
     CAST(report_ahs_mtemplate_deduped.encounterID AS BIGINT)
 AND CAST(report_ahs_chart_viewer_deduped.DocumentTypeID AS BIGINT) =
     CAST(report_ahs_mtemplate_deduped.DocumentTypeDE AS BIGINT)
 AND report_ahs_chart_viewer_deduped.row_number = 1

LEFT JOIN dbo_vendor_item_deduped
  ON CAST(dbo_vendor_item_deduped.ID AS BIGINT) =
     CAST(report_ahs_mtemplate_deduped.ID AS BIGINT)
 AND dbo_vendor_item_deduped.row_number = 1

LEFT JOIN dbo_vendor_item_extension_deduped
  ON CAST(dbo_vendor_item_extension_deduped.VendorItemID AS BIGINT) =
     CAST(dbo_vendor_item_deduped.ID AS BIGINT)

JOIN _exponent.omop_mapping.source_to_person
  ON source_to_person.person_source_value = CONCAT_WS(
       CHR(31),
       'allscripts_tw',
       'dbo_person',
       'id',
       CAST(report_ahs_mtemplate_deduped.patientid AS BIGINT)
     )
 AND source_to_person.active_flag = TRUE

LEFT JOIN _exponent.omop_mapping.source_to_provider
  ON source_to_provider.provider_source_value = CONCAT_WS(
       CHR(31),
       'allscripts_tw',
       'dbo_provider',
       'id',
       CAST(report_ahs_mtemplate_deduped.WhoDidItID AS BIGINT)
     )
 AND source_to_provider.active_flag = TRUE

JOIN _exponent.omop_mapping.source_to_visit_occurrence
  ON source_to_visit_occurrence.visit_occurrence_source_value = CONCAT_WS(
       CHR(31),
       'allscripts_tw',
       'dbo_visit',
       'id',
       CAST(dbo_visit.id AS BIGINT)
     )
 AND source_to_visit_occurrence.active_flag = TRUE

WHERE report_ahs_mtemplate_deduped.row_number = 1
  AND (
       report_ahs_mtemplate_deduped.answer IS NOT NULL
       OR dbo_vendor_item_extension_deduped.ExtensionValue IS NOT NULL
  );

In [0]:
%sql
MERGE INTO _exponent.omop_silver.note AS target
USING silver_note AS source
ON target.note_source_value = source.note_source_value

WHEN MATCHED AND (
     NOT (target.person_id <=> source.person_id)
  OR NOT (target.note_date <=> source.note_date)
  OR NOT (target.note_datetime <=> source.note_datetime)
  OR NOT (target.note_type_concept_id <=> source.note_type_concept_id)
  OR NOT (target.note_class_concept_id <=> source.note_class_concept_id)
  OR NOT (target.note_title <=> source.note_title)
  OR NOT (target.note_text <=> source.note_text)
  OR NOT (target.encoding_concept_id <=> source.encoding_concept_id)
  OR NOT (target.language_concept_id <=> source.language_concept_id)
  OR NOT (target.provider_id <=> source.provider_id)
  OR NOT (target.visit_occurrence_id <=> source.visit_occurrence_id)
  OR NOT (target.visit_detail_id <=> source.visit_detail_id)
  OR NOT (target.note_event_id <=> source.note_event_id)
  OR NOT (target.note_event_field_concept_id <=> source.note_event_field_concept_id)
  OR NOT (target.source_system <=> source.source_system)
)
THEN UPDATE SET
  target.person_id = source.person_id,
  target.note_date = source.note_date,
  target.note_datetime = source.note_datetime,
  target.note_type_concept_id = source.note_type_concept_id,
  target.note_class_concept_id = source.note_class_concept_id,
  target.note_title = source.note_title,
  target.note_text = source.note_text,
  target.encoding_concept_id = source.encoding_concept_id,
  target.language_concept_id = source.language_concept_id,
  target.provider_id = source.provider_id,
  target.visit_occurrence_id = source.visit_occurrence_id,
  target.visit_detail_id = source.visit_detail_id,
  target.note_event_id = source.note_event_id,
  target.note_event_field_concept_id = source.note_event_field_concept_id,
  target.source_system = source.source_system,
  target.last_mod_tsp = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  note_source_value,
  person_id,
  note_date,
  note_datetime,
  note_type_concept_id,
  note_class_concept_id,
  note_title,
  note_text,
  encoding_concept_id,
  language_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  note_event_id,
  note_event_field_concept_id,
  source_system,
  last_mod_tsp
)
VALUES (
  source.note_source_value,
  source.person_id,
  source.note_date,
  source.note_datetime,
  source.note_type_concept_id,
  source.note_class_concept_id,
  source.note_title,
  source.note_text,
  source.encoding_concept_id,
  source.language_concept_id,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.note_event_id,
  source.note_event_field_concept_id,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_note (
    source_system,
    note_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    source.source_system,
    source.note_source_value,
    TRUE                AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    CURRENT_TIMESTAMP() AS last_mod_tsp,
    NULL                AS merge_id,
    NULL                AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        note_source_value
    FROM silver_note
    WHERE note_source_value IS NOT NULL
      AND source_system = 'allscripts_tw'
) source
LEFT ANTI JOIN _exponent.omop_mapping.source_to_note target
  ON target.note_source_value = source.note_source_value
 AND target.source_system = source.source_system;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold AS
SELECT
  source_to_note.note_id,

  note.person_id,

  note.note_date,
  note.note_datetime,
  note.note_type_concept_id,
  note.note_class_concept_id,
  note.note_title,
  note.note_text,
  note.encoding_concept_id,
  note.language_concept_id,
  note.provider_id,

  note.visit_occurrence_id,

  note.visit_detail_id,
  note.note_event_id,
  note.note_event_field_concept_id,
  note.note_source_value

FROM _exponent.omop_silver.note

JOIN _exponent.omop_mapping.source_to_note
  ON source_to_note.note_source_value = note.note_source_value
 AND source_to_note.source_system = 'allscripts_tw'
 AND source_to_note.active_flag = TRUE

WHERE note.source_system = 'allscripts_tw';

In [0]:
%sql
MERGE INTO _exponent.omop_tw.note AS target
USING gold AS source
ON target.note_id = source.note_id

WHEN MATCHED AND NOT (
     target.person_id             <=> source.person_id
 AND target.note_date             <=> source.note_date
 AND target.note_datetime         <=> source.note_datetime
 AND target.note_type_concept_id  <=> source.note_type_concept_id
 AND target.note_class_concept_id <=> source.note_class_concept_id
 AND target.note_title            <=> source.note_title
 AND target.note_text             <=> source.note_text
 AND target.encoding_concept_id   <=> source.encoding_concept_id
 AND target.language_concept_id   <=> source.language_concept_id
 AND target.provider_id           <=> source.provider_id
 AND target.visit_occurrence_id   <=> source.visit_occurrence_id
 AND target.visit_detail_id             <=> source.visit_detail_id
 AND target.note_event_id               <=> source.note_event_id
 AND target.note_event_field_concept_id <=> source.note_event_field_concept_id
 AND target.note_source_value     <=> source.note_source_value
) THEN UPDATE SET
  target.person_id             = source.person_id,
  target.note_date             = source.note_date,
  target.note_datetime         = source.note_datetime,
  target.note_type_concept_id  = source.note_type_concept_id,
  target.note_class_concept_id = source.note_class_concept_id,
  target.note_title            = source.note_title,
  target.note_text             = source.note_text,
  target.encoding_concept_id   = source.encoding_concept_id,
  target.language_concept_id   = source.language_concept_id,
  target.provider_id           = source.provider_id,
  target.visit_occurrence_id   = source.visit_occurrence_id,
  target.visit_detail_id             = source.visit_detail_id,
  target.note_event_id               = source.note_event_id,
  target.note_event_field_concept_id = source.note_event_field_concept_id,
  target.note_source_value     = source.note_source_value

WHEN NOT MATCHED THEN INSERT (
  note_id,
  person_id,
  note_date,
  note_datetime,
  note_type_concept_id,
  note_class_concept_id,
  note_title,
  note_text,
  encoding_concept_id,
  language_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  note_event_id,
  note_event_field_concept_id,
  note_source_value
) VALUES (
  source.note_id,
  source.person_id,
  source.note_date,
  source.note_datetime,
  source.note_type_concept_id,
  source.note_class_concept_id,
  source.note_title,
  source.note_text,
  source.encoding_concept_id,
  source.language_concept_id,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.note_event_id,
  source.note_event_field_concept_id,
  source.note_source_value
);

In [0]:
%sql
MERGE INTO _exponent.omop_allscripts.note AS target
USING gold AS source
ON target.note_id = source.note_id

WHEN MATCHED AND NOT (
     target.person_id             <=> source.person_id
 AND target.note_date             <=> source.note_date
 AND target.note_datetime         <=> source.note_datetime
 AND target.note_type_concept_id  <=> source.note_type_concept_id
 AND target.note_class_concept_id <=> source.note_class_concept_id
 AND target.note_title            <=> source.note_title
 AND target.note_text             <=> source.note_text
 AND target.encoding_concept_id   <=> source.encoding_concept_id
 AND target.language_concept_id   <=> source.language_concept_id
 AND target.provider_id           <=> source.provider_id
 AND target.visit_occurrence_id   <=> source.visit_occurrence_id
 AND target.visit_detail_id             <=> source.visit_detail_id
 AND target.note_event_id               <=> source.note_event_id
 AND target.note_event_field_concept_id <=> source.note_event_field_concept_id
 AND target.note_source_value     <=> source.note_source_value
) THEN UPDATE SET
  target.person_id             = source.person_id,
  target.note_date             = source.note_date,
  target.note_datetime         = source.note_datetime,
  target.note_type_concept_id  = source.note_type_concept_id,
  target.note_class_concept_id = source.note_class_concept_id,
  target.note_title            = source.note_title,
  target.note_text             = source.note_text,
  target.encoding_concept_id   = source.encoding_concept_id,
  target.language_concept_id   = source.language_concept_id,
  target.provider_id           = source.provider_id,
  target.visit_occurrence_id   = source.visit_occurrence_id,
  target.visit_detail_id             = source.visit_detail_id,
  target.note_event_id               = source.note_event_id,
  target.note_event_field_concept_id = source.note_event_field_concept_id,
  target.note_source_value     = source.note_source_value

WHEN NOT MATCHED THEN INSERT (
  note_id,
  person_id,
  note_date,
  note_datetime,
  note_type_concept_id,
  note_class_concept_id,
  note_title,
  note_text,
  encoding_concept_id,
  language_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  note_event_id,
  note_event_field_concept_id,
  note_source_value
) VALUES (
  source.note_id,
  source.person_id,
  source.note_date,
  source.note_datetime,
  source.note_type_concept_id,
  source.note_class_concept_id,
  source.note_title,
  source.note_text,
  source.encoding_concept_id,
  source.language_concept_id,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.note_event_id,
  source.note_event_field_concept_id,
  source.note_source_value
);